In [ ]:
import kagglehub
dataset = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print(dataset)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Memory growth enabled on {len(gpus)} GPU(s)")
else:
    print("No GPU found, running on CPU")

In [ ]:
IMG_SIZE = 32
PATCH_SIZE = 4
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4
MLP_HEAD_UNITS = [128, 64]
BATCH_SIZE = 256
EPOCHS = 25
LEARNING_RATE = 3e-4

TRAIN_DIR = dataset + "/train"
TEST_DIR  = dataset + "/test"
print("TRAIN:", TRAIN_DIR)
print("TEST: ", TEST_DIR)

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_gen = ImageDataGenerator(rescale=1./255)

train_loader = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)
test_loader = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)
print("Classes:", train_loader.class_indices)
print("Train batches:", len(train_loader), "  Test batches:", len(test_loader))

In [ ]:
def mlp_block(x, hidden_units, dropout_rate=0.1):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


class PatchExtractor(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

In [ ]:
def build_cnn_vit_hybrid(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    num_patches=NUM_PATCHES,
    projection_dim=PROJECTION_DIM,
    num_heads=NUM_HEADS,
    transformer_layers=TRANSFORMER_LAYERS,
    mlp_head_units=MLP_HEAD_UNITS,
):
    inputs = keras.Input(shape=(img_size, img_size, 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    cnn_features = layers.GlobalAveragePooling2D()(x)

    patches = PatchExtractor(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attn_output, encoded_patches])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp_block(x3, [projection_dim * 2, projection_dim])
        encoded_patches = layers.Add()([x3, x2])

    vit_output = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    vit_features = layers.Flatten()(vit_output)
    vit_features = layers.Dense(projection_dim * 2, activation=tf.nn.gelu)(vit_features)
    vit_features = layers.Dropout(0.3)(vit_features)

    combined = layers.Concatenate()([cnn_features, vit_features])

    out = mlp_block(combined, mlp_head_units, dropout_rate=0.3)
    outputs = layers.Dense(1, activation="sigmoid")(out)

    model = keras.Model(inputs=inputs, outputs=outputs, name="CNN_ViT_Hybrid")
    return model


model = build_cnn_vit_hybrid()
model.summary()

In [ ]:
lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LEARNING_RATE,
    first_decay_steps=len(train_loader) * 5,
    t_mul=2.0,
    m_mul=0.9,
)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        "cnn_vit_hybrid_best.keras", monitor="val_accuracy", save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

In [ ]:
history = model.fit(
    train_loader,
    validation_data=test_loader,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss Curve")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_true = test_loader.classes
y_probs = model.predict(test_loader).ravel()
y_pred = (y_probs > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["FAKE", "REAL"], yticklabels=["FAKE", "REAL"])
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - CNN+ViT Hybrid")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
model.save("cnn_vit_hybrid_fake_detector.keras")
print("Model saved as cnn_vit_hybrid_fake_detector.keras")